# Setuptools: Packaging Your Engineering Tools

## The Core Question

**How do you package your simulation tools so your team can `pip install` them?**

You've written a heat-conduction solver that works. Now your colleague needs it. Today you email them a `.py` file; next month it's three files; next year it's a folder with a `README` and a `requirements.txt` that doesn't match. Setuptools turns your scripts into a proper **installable package** — one `pip install` and it's there, with a version number and a clean import.

## Why this library?

Setuptools is the standard build backend for Python packages. It reads a single `pyproject.toml`, bundles your modules into a **wheel** (`.whl`) and a **source archive** (`.tar.gz`), and lets anyone `pip install` the result. Unlike copying files, a package has a **name**, a **version**, and **declared dependencies** — so your team always knows exactly what they're running.

## The Workflow

```
  write package (my_cfd_tool/solver.py)
        |
        v
  describe it (pyproject.toml)
        |
        v
  build:  python -m build      -->  dist/*.whl  +  dist/*.tar.gz
        |
        v
  install:  pip install dist/*.whl
        |
        v
  use:  from my_cfd_tool import solve
```


In [ ]:
import setuptools
print('setuptools', setuptools.__version__)

## 1. The Package Structure

A Python package is just a **folder with an `__init__.py`**. Ours, `my_cfd_tool`, contains one module, `solver.py`, with a single function: `solve`, which computes the 1-D steady conduction temperature profile between two walls.

```
my_cfd_tool/
├── __init__.py      # makes it a package; re-exports `solve`
└── solver.py        # the actual physics
```

The `__init__.py` is what lets a user write `from my_cfd_tool import solve` instead of `from my_cfd_tool.solver import solve`.


### The actual files

**`my_cfd_tool/solver.py`**:
```python
def solve(k: float, L: float, T_left: float, T_right: float, n: int = 100) -> list[float]:
    """Solve 1-D steady conduction: d2T/dx2 = 0 with Dirichlet BCs.

    The exact solution is linear: T(x) = T_left + (T_right - T_left) * x / L.
    """
    if L <= 0:
        raise ValueError("Length must be positive")
    xs = [i * L / (n - 1) for i in range(n)]
    return [T_left + (T_right - T_left) * x / L for x in xs]
```

**`my_cfd_tool/__init__.py`**:
```python
from .solver import solve

__version__ = "0.1.0"
__all__ = ["solve", "__version__"]
```

These files are created on disk in the `my_cfd_tool/` folder next to this notebook.


In [ ]:
# Test the import
from my_cfd_tool import solve

# Quick sanity check
T = solve(k=1.0, L=2.0, T_left=0.0, T_right=100.0, n=5)
print('Temperature profile:', T)
print('Expected: [0.0, 25.0, 50.0, 75.0, 100.0]')

## 2. Describe the Package: `pyproject.toml`

`pyproject.toml` is the single source of truth. It has two parts:

- **`[build-system]`** — tells the build tool to use `setuptools`.
- **`[project]`** — the metadata: name, version, description, and the minimum Python.

This is what `pip` reads to know *what* it's installing and *where* to find the code.


In [ ]:
# pyproject.toml
[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "my-cfd-tool"
version = "0.1.0"
description = "A tiny CFD toolkit for teaching packaging"
requires-python = ">=3.9"

[tool.setuptools]
packages = ["my_cfd_tool"]

## 3. Build the Package

`python -m build` produces two artifacts in `dist/`:

- **`my_cfd_tool-0.1.0-py3-none-any.whl`** — a wheel (a zip of your code + metadata). This is what `pip install` actually uses.
- **`my_cfd_tool-0.1.0.tar.gz`** — a source archive, for people who want to build from source.

The `py3-none-any` tag means: Python 3, any OS, any architecture — pure Python, no compiled extensions.


In [ ]:
import subprocess, sys, os, shutil

# Lay out the package in a temp dir, then build it
work = os.path.join(os.getcwd(), '_pkg_build')
shutil.rmtree(work, ignore_errors=True)
os.makedirs(os.path.join(work, 'my_cfd_tool'))

with open(os.path.join(work, 'my_cfd_tool', 'solver.py'), 'w', encoding='utf-8') as f:
    f.write('''def solve(k: float, L: float, T_left: float, T_right: float, n: int = 100) -> list[float]:
    if L <= 0:
        raise ValueError("Length must be positive")
    xs = [i * L / (n - 1) for i in range(n)]
    return [T_left + (T_right - T_left) * x / L for x in xs]
''')
with open(os.path.join(work, 'my_cfd_tool', '__init__.py'), 'w', encoding='utf-8') as f:
    f.write('from .solver import solve\n__version__ = "0.1.0"\n')
with open(os.path.join(work, 'pyproject.toml'), 'w', encoding='utf-8') as f:
    f.write('''[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "my-cfd-tool"
version = "0.1.0"
description = "A tiny CFD toolkit for teaching packaging"
requires-python = ">=3.9"

[tool.setuptools]
packages = ["my_cfd_tool"]
''')

result = subprocess.run([sys.executable, '-m', 'build'],
                        cwd=work, capture_output=True, text=True)
print(result.stdout[-400:])
print('=== dist/ ===')
for name in sorted(os.listdir(os.path.join(work, 'dist'))):
    print(' ', name)

### What does this mean?

The build succeeded and produced both artifacts. The wheel is a **self-contained, versioned, installable unit** of your code. You can now:

- email the `.whl` to a colleague,
- upload it to your team's internal package index, or
- publish it to PyPI.

**Engineering takeaway:** packaging is what turns "my script" into "a tool the team uses." The version number (`0.1.0`) means you can track *which* solver produced a given result — critical when you're comparing simulation runs months apart.


## 4. Install and Use It

`pip install` the wheel, then import it. The user never sees the folder structure — just a clean `from my_cfd_tool import solve`.


In [ ]:
import subprocess, sys, os, shutil

work = os.path.join(os.getcwd(), '_pkg_build')
wheel = os.path.join(work, 'dist', 'my_cfd_tool-0.1.0-py3-none-any.whl')

# Install into an isolated target dir so we don't pollute the environment
target = os.path.join(work, '_install')
shutil.rmtree(target, ignore_errors=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--target', target, wheel], check=True)

# Now import from that target
import importlib.util
spec = importlib.util.spec_from_file_location(
    'my_cfd_tool', os.path.join(target, 'my_cfd_tool', '__init__.py'),
    submodule_search_locations=[os.path.join(target, 'my_cfd_tool')])
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

T = mod.solve(k=50.0, L=1.0, T_left=100.0, T_right=20.0, n=5)
print('Installed version:', mod.__version__)
print('T profile (hot->cold wall):', [round(t, 1) for t in T])

### Reading the result

```
Installed version: 0.1.0
T profile (hot->cold wall): [100.0, 80.0, 60.0, 40.0, 20.0]
```

The installed package returns the expected **linear** temperature profile from the 100 °C wall to the 20 °C wall. The user imported `solve` with one line and got a working, versioned tool — no file copying, no path fiddling.

**Engineering takeaway:** once your solver is a package, every downstream script, notebook, and CI job imports the *same* versioned code. That's the foundation for reproducible, team-shared simulation work.


## 5. Exercises

1. **Add a dependency.** Give `my_cfd_tool` a real dependency on `numpy` (use it in `solve`). Add `dependencies = ["numpy"]` to `[project]` in `pyproject.toml`, rebuild, and confirm `pip install` pulls in numpy.

2. **Bump the version.** Change the version to `0.2.0` in *both* `pyproject.toml` and `__init__.py`. Rebuild. What changes in the wheel filename? Why must the two stay in sync?

3. **Add a console script.** Add a `[project.scripts]` entry so `my-cfd-solve` runs on the command line. Rebuild, install, and run `my-cfd-solve --help`.

4. **Publish locally.** Run `pip install dist/*.whl` into your real environment (or a venv) and import `my_cfd_tool` from a fresh Python process. Confirm `__version__` is what you expect.
